# [실습] 벡터 데이터베이스 기반 RAG 어플리케이션 만들기

RAG는 Retrieval-Augmented Generation (RAG) 의 약자로,   
LLM의 작동 과정에 검색을 결합하여 답변 성능을 높이는 어플리케이션입니다.   


이번 실습에서는 뉴스 검색 데이터를 이용한 RAG를 수행해 보겠습니다.    
코랩 GPU 사용을 위한 설정이 필요합니다.

### <필수> 실습을 진행하기 전, GPU를 T4로 설정해 주세요!

## 라이브러리 설치  

랭체인 관련 라이브러리와 벡터 데이터베이스 라이브러리를 설치합니다.  
<br>

`sentence_transformers`: 트랜스포머 계열의 공개 임베딩 모델을 사용할 수 있습니다.    
`langchain_chroma`: ChromaDB를 이용해 벡터 데이터베이스를 구성합니다.

In [1]:
%pip install dotenv langchain_huggingface transformers sentence_transformers jsonlines langchain langchain-openai langchain-community langchain_chroma -q

Note: you may need to restart the kernel to use updated packages.


코랩에서 실행 시, Restart 메시지가 나타납니다. 런타임을 재시작하여 패키지를 정리합니다.

## LLM과 임베딩 모델 구성하기   

이번 실습에서는 LLM 모델과 함께 임베딩 모델이 필요합니다.   
임베딩 모델은 텍스트를 벡터로 변환하며,    
이후 결과를 벡터 DB에 저장해 검색할 수 있습니다.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 확인')

OpenAI API 키 확인


In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.6", reasoning_effort='low')
llm.invoke("hi")

c:\apps\RAG2026\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AIMessage(content='Hi! How can I help?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 7, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-sol', 'system_fingerprint': None, 'id': 'chatcmpl-E8lrFaR7q4tAWHub0E7CjbV7w7wYg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fc78c-839d-7243-9eed-515663e6141f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 10, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

OpenAI의 `text-embedding-3-large` 는 빠른 속도로 연산이 가능하나, 비용이 발생하며 온라인 모델입니다.   
이에 따라, 폐쇄망/온프레미스 환경에서는 공개 임베딩 모델을 사용하여 구현해야 합니다.

In [4]:
from langchain_openai import OpenAIEmbeddings
openai_embeddings = OpenAIEmbeddings(model='text-embedding-3-large')
# 24.1

허깅페이스에 게시된 공개 모델을 불러옵니다.   
오픈 임베딩 모델에서 중요한 파라미터는 다음과 같습니다.

- 파라미터 수 : 큰 임베딩 모델의 크기는 LLM에 육박합니다. GPU를 고려하여 선택합니다.
- Max Tokens: 임베딩 모델의 최대 토큰보다 큰 데이터를 입력하면, 앞부분만을 이용해 계산하게 되므로 적절한 검색이 되지 않을 수 있습니다.
- 임베딩 차원: 큰 차원의 벡터를 생성하는 임베딩 모델은 검색 속도가 감소합니다.

현재 한국어 데이터를 임베딩하기 위해 자주 사용하는 모델은 아래와 같습니다.


- Qwen/Qwen-3-Embedding (0.6B, 4B, 8B, 32768 토큰 제한)    
알리바바 클라우드의 Qwen 모델을 개량하여 만든 모델입니다.    
가장 최신 모델로, BGE-M3와의 성능 비교가 치열합니다.


Qwen 3 임베딩 모델을 불러옵니다.

In [5]:
# 터미널에서 아래 코드를 실행해도 됨
# hf download Qwen/Qwen3-Embedding-0.6B --local-dir ./embedding

In [12]:
from sentence_transformers import SentenceTransformer
import torch

# HuggingFace 임베딩 주소 지정하기
# intfloat/multilingual-e5-small , baai/bge-m3, 등의 주소를 입력하여 지정
# GPU에 여유가 있다면 Qwen3 Embedding의 큰 사이즈 (4B, 8B)

model_name = 'Qwen/Qwen3-Embedding-0.6B'
#실제 주소: https://huggingface.co/Qwen/Qwen3-Embedding-0.6B

# CPU 설정으로 모델 불러오기
emb_model = SentenceTransformer(model_name, device='cpu',model_kwargs={'torch_dtype':torch.bfloat16})

# 로컬 폴더에 모델 저장하기
emb_model.save('./embedding')

# 모델 메모리에서 삭제
del emb_model
import gc
gc.collect()

print("임베딩 모델 저장 완료!")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s]


임베딩 모델 저장 완료!


파일 시스템에 저장한 오픈 모델은 HuggingFaceEmbeddings로 불러옵니다.

In [13]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

# CPU를 사용하도록 device 설정 변경 및 torch_dtype 조정
open_embeddings = HuggingFaceEmbeddings(
    model_name='./embedding',
    model_kwargs={
        'device': 'cpu',
        'model_kwargs': {'torch_dtype': torch.float32}  # CPU 환경에서는 float32 권장
    }
)

# CPU 로드 확인
print("임베딩 모델 CPU 로드 완료")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 975.04it/s] 


임베딩 모델 CPU 로드 완료


RAG를 하기 전, 비교를 위해 LLM에게 질문해 보겠습니다.

In [14]:
# Test
llm.invoke("도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?")

AIMessage(content='**도메인 특화 언어 모델(Domain-Specific Language Model)**은 특정 산업·업무·지식 영역의 데이터를 중심으로 학습하거나 조정한 언어 모델입니다. 범용 언어 모델보다 해당 분야의 **전문 용어, 문서 형식, 업무 규칙, 표현 방식**을 더 정확하게 이해하고 생성하는 것이 목적입니다.\n\n### 만드는 방식\n- **도메인 사전학습**: 의료 논문, 법률 문서, 금융 보고서 등으로 추가 학습\n- **미세조정(Fine-tuning)**: 특정 업무의 질문·답변이나 문서 데이터로 조정\n- **RAG**: 사내 문서나 전문 데이터베이스를 검색해 답변에 활용\n- **규칙·도구 연동**: 계산기, 코드 실행기, 의료·법률 시스템 등과 연결\n\n### 대표적인 예시\n\n| 분야 | 모델·시스템 예시 | 주요 용도 |\n|---|---|---|\n| 의료·바이오 | **BioBERT, ClinicalBERT, Med-PaLM** | 논문 검색, 임상 기록 분석, 의료 질의응답 |\n| 금융 | **FinBERT, BloombergGPT** | 금융 감성 분석, 시장·기업 문서 분석 |\n| 법률 | **Legal-BERT, Harvey** | 판례 검색, 계약서 검토, 법률 문서 작성 지원 |\n| 과학 | **SciBERT, Galactica** | 과학 문헌 분류·검색·요약 |\n| 프로그래밍 | **Code Llama, StarCoder, GitHub Copilot 기반 모델** | 코드 생성, 설명, 오류 수정 |\n| 기업 내부 | 사내 문서로 구축한 전용 LLM/RAG | 업무 규정 질의응답, 고객 지원, 보고서 작성 |\n\n### 장점\n- 전문 용어와 맥락을 더 잘 처리\n- 특정 업무에서 정확도와 일관성이 높음\n- 조직의 문서 형식이나 정책을 반영할 수 있음\n- 작은 모델로도 제한된 영역에서는 높은 효율을 낼 수 있음\n\n### 한계\n- 학습 범위 밖의 질문에는 성능이 떨어질 수 

## 데이터 준비하기    
네이버 API를 통해, 검색어에 대한 뉴스 기사 링크를 가져오겠습니다.    


In [15]:
import requests
def get_naver_news_links(query, num_links=100):
    """
    query와 num_links를 입력받아 네이버 검색 수행, 네이버 뉴스 URL의 기사만 수집
    """

    url = f"https://openapi.naver.com/v1/search/news.json?query={query}&display={num_links}&sort=sim"
    # 최대 100개의 결과를 표시
    headers = {
        'X-Naver-Client-Id': 'Ko6yIqbV2TOHq9rPH8tu',
        'X-Naver-Client-Secret': 'BvqX8mNtHu'
    }

    response = requests.get(url, headers=headers)
    result = response.json()
    # 특정 링크 형식만 필터링
    filtered_links = []
    for item in result['items']:
        link = item['link']
        if "n.news.naver.com/mnews/article/" in link:
            # 네이버 뉴스 스타일만 모으기
            filtered_links.append(link)

    # 결과 출력
    print(query, ':', len(filtered_links), 'Example:', filtered_links[0])
    # for link in filtered_links:
    #     print(link)

    return filtered_links

filtered_links = []
for topic in ['도메인 특화 언어모델', 'OpenAI', 'GPT', '구글', '가전제품', '넷플릭스']:
    filtered_links += get_naver_news_links(topic, 100)
print('Total Articles:', len(filtered_links))
print('Total Articles(Without Duplicate):',len(list(set(filtered_links))))
filtered_links = list(set(filtered_links))

도메인 특화 언어모델 : 40 Example: https://n.news.naver.com/mnews/article/029/0003038168?sid=105
OpenAI : 21 Example: https://n.news.naver.com/mnews/article/030/0003453737?sid=101
GPT : 76 Example: https://n.news.naver.com/mnews/article/001/0016229604?sid=104
구글 : 71 Example: https://n.news.naver.com/mnews/article/003/0014104826?sid=105
가전제품 : 46 Example: https://n.news.naver.com/mnews/article/011/0004647790?sid=101
넷플릭스 : 34 Example: https://n.news.naver.com/mnews/article/092/0002432776?sid=105
Total Articles: 288
Total Articles(Without Duplicate): 287


## LangChain Document Loaders

LangChain의 `document_loaders`는 다양한 형식의 파일을 불러올 수 있습니다.   
[https://python.langchain.com/docs/integrations/document_loaders/ ]    

Web URL로부터 페이지를 로드하는 기본 파서인 `WebBaseLoader`를 사용합니다.   

In [20]:
# # Jupyter 분산 처리를 위한 설정 (코랩에서는 불필요)
import nest_asyncio

nest_asyncio.apply()

In [23]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

async def get_news_documents(links):
    loader = WebBaseLoader(
        web_paths=links,
        bs_kwargs={'parse_only':bs4.SoupStrainer(class_=("newsct", "newsct-body"))},
                                # newsct, newsct-body만 추출 : 네이버 뉴스 포맷 HTML 요소

        requests_per_second = 10, # 1초에 10개 요청 보내기
        show_progress = True # 진행 상황 출력
    )
    # docs = loader.load() # 기본 코드

    docs = []

    async for doc in loader.alazy_load(): # 순차적 로드 대신 비동기 처리
        docs.append(doc)
    return docs

docs = await get_news_documents(filtered_links)

C:\Users\김수현\AppData\Local\Temp\ipykernel_20768\2711377856.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.
Fetching pages: 100%|##########| 287/287 [00:08<00:00, 32.37it/s]


In [24]:
docs[12]

Document(metadata={'source': 'https://n.news.naver.com/mnews/article/016/0002676672?sid=103'}, page_content='\n\n\n\n\n\n\n헤럴드경제\n\n\n구독\n\n헤럴드경제 언론사 구독되었습니다. 메인 뉴스판에서  주요뉴스를  볼 수 있습니다.\n보러가기\n닫기\n\n\n헤럴드경제 언론사 구독 해지되었습니다.\n닫기\n\n\n\n\n‘참교육’, 공개 8주만에 넷플릭스 역대 비영어 쇼 10위\n\n\n\n\n\n\n이명수 기자\n\n\n\n\n\n이명수 기자\n\n\n\n\n\n\n입력\n2026.07.29. 오전 10:41\n\n\n\n기사원문\n \n\n\n\n\n\n\n\n\n\n\n추천\n반응\n\n\n\n\n쏠쏠정보\n0\n\n\n\n\n흥미진진\n0\n\n\n\n\n공감백배\n0\n\n\n\n\n분석탁월\n0\n\n\n\n\n후속강추\n0\n\n\n \n\n\n\n\n댓글\n반응\n\n\n\n\n\n\n\n\n텍스트 음성 변환 서비스 사용하기\n\n\n\n성별\n남성\n여성\n\n\n말하기 속도\n느림\n보통\n빠름\n\n이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다.\n본문듣기 시작\n\n닫기\n\n\n \n\n글자 크기 변경하기\n\n글자크기\n\n\n가1단계\n작게\n\n\n가2단계\n보통\n\n\n가3단계\n크게\n\n\n가4단계\n아주크게\n\n\n가5단계\n최대크게\n\n\n닫기\n\n\n\n\nSNS 보내기\n\n\n\n인쇄하기\n\n\n\n\n\n\n\n\n\n\n\n\n넷플릭스 드라마 ‘참교육’ 스틸컷 [넷플릭스 제공][헤럴드경제=이명수 기자] ‘참교육’이 ‘오징어 게임’에 이어 넷플릭스 역대 최고 인기 비영어 쇼 톱 10에 들었다.연합뉴스에 따르면, 넷플릭스는 ‘참교육’이 공개 8주 만에 시청수(Views·시청 시간을 러닝타임으로 나눈 값) 6천20만을 기록하며 넷플릭스 비영어권 역대 최고 인기 시리즈 10위에 

크롤링 결과에는 불필요한 문자가 많이 포함되어 있습니다.    
전처리를 통해 이를 제거합니다.

In [25]:
import re

def preprocess(docs):
    noise_texts = [
        '''구독중 구독자 0 응원수 0 더보기''',
        '''쏠쏠정보 0 흥미진진 0 공감백배 0 분석탁월 0 후속강추 0''',
        '''댓글 본문 요약봇 본문 요약봇''',
        '''도움말 자동 추출 기술로 요약된 내용입니다. 요약 기술의 특성상 본문의 주요 내용이 제외될 수 있어, 전체 맥락을 이해하기 위해서는 기사 본문 전체보기를 권장합니다. 닫기''',
        '''텍스트 음성 변환 서비스 사용하기 성별 남성 여성 말하기 속도 느림 보통 빠름''',
        '''이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다. 본문듣기 시작''',
        '''닫기 글자 크기 변경하기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 SNS 보내기 인쇄하기''',
        'PICK 안내 언론사가 주요기사로선정한 기사입니다. 언론사별 바로가기 닫기',
        '응원 닫기',
        '구독 구독중 구독자 0 응원수 0 ',
    ]

    def clean_text(doc):
        text = doc.page_content
        # 탭과 개행문자를 공백으로 변환
        text = text.replace('\t', ' ').replace('\n', ' ')

        # 연속된 공백을 하나로 치환
        text = re.sub(r'\s+', ' ', text).strip()

        # 여러 구분자를 한번에 처리
        split_markers = [
            '구독 해지되었습니다.',
            '구독 메인에서 바로 보는 언론사 편집 뉴스 지금 바로 구독해보세요!'
        ]
        for marker in split_markers:
            parts = text.split(marker)
            if len(parts) > 1:
                if marker == '구독 해지되었습니다.':
                    text = parts[1]  # 뒷부분 사용
                else:
                    text = parts[0]  # 앞부분 사용

        # 노이즈 텍스트 제거
        for noise in noise_texts:
            text = text.replace(noise, '')

        # 연속된 공백을 하나로 치환
        text = re.sub(r'\s+', ' ', text).strip()
        doc.page_content = text
        return doc

    preprocessed_docs = []
    for doc in docs:

        # 텍스트 정제
        doc= clean_text(doc)
        preprocessed_docs.append(doc)

    return preprocessed_docs

preprocessed_docs = preprocess(docs)


In [26]:
preprocessed_docs[2]

Document(metadata={'source': 'https://n.news.naver.com/mnews/article/586/0000135023?sid=103'}, page_content='닫기 K콘텐츠 대국의 그림자…\'넷플릭스 10년\'이 만든 황금기의 역설 조유빈 기자 조유빈 기자 입력 2026.08.01. 오후 4:00 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 [조유빈 기자 you@sisajournal.com] 세계에 K콘텐츠 각인시킨 파트너…대규모 투자로 경쟁력 높여압도적 자본, 韓 업계 위협…장기적 성장 위한 IP 확보도 과제과거 우리는 넷플릭스를 완전히 오판했다. 정부 산하 한 공공기관은 넷플릭스의 국내 진출을 앞두고 이렇게 전망했다. "한국 미디어 시장 판도를 바꿀 정도의 변화를 끌어낼 수 있다고 보는 전문가는 거의 없다. 한국의 이용자 중 넷플릭스를 월정액으로 이용하는 사람은 많지 않을 것이다. 넷플릭스는 적은 예산을 투입해 소비자 반응을 끌어내는 시도를 하는 것이 바람직할 수 있다." 그러나 10년이 지난 지금, 모든 전망은 뒤집혔다. 판도를 바꾸지 못할 것이라던 플랫폼은 한국 미디어 생태계의 중심이 됐고, OTT 시장 1위를 굳건히 지키고 있다. 막대한 금액을 투자한 넷플릭스는 K콘텐츠를 세계 정상에 올려놓은 파트너임과 동시에 국내 콘텐츠 산업의 구조를 흔드는 최대 변수로 자리 잡았다.넷플릭스는 국내 산업만으로는 이루기 어려웠던 성과를 한국에 안겼지만, 그것은 온전히 한국의 것이 되진 못했다. 히트작이 나올수록 넷플릭스의 콘텐츠 자산은 늘어났지만 한국 제작 현장에는 IP와 자본이 그만큼 축적되지 못했기 때문이다. 성공의 결과물이 상당 부분 글로벌 플랫폼으로 흘러가면서 한국의 콘텐츠 황금기는 아이러니하게도 외형은 호황, 내실은 미완인 반쪽짜리 번영으로 귀결되고 있다. 넷플릭스를 오판했다는 사실을 다시 상기하는

불러온 텍스트 데이터는 파일로 저장할 수 있습니다.

In [27]:
# 불러온 document 저장하기

import jsonlines
def save_docs_to_jsonl(documents, file_path):
    with jsonlines.open(file_path, mode="w") as writer:
        for doc in documents:
            writer.write(doc.model_dump())

# jsonl 파일 불러오기
from langchain_core.documents import Document

def load_docs_from_jsonl(file_path):
    documents = []
    with jsonlines.open(file_path, mode="r") as reader:
        for doc in reader:
            documents.append(Document(**doc))
    return documents

In [28]:
# 저장
save_docs_to_jsonl(preprocessed_docs, "docs.jsonl")

## Chunking: 청크 단위로 나누기   



전처리가 완료된 docs를 chunk 단위로 분리합니다.
`chunk_size`와 `chunk_overlap`을 이용해 청크의 구성 방식을 조절할 수 있습니다.

Chunk Size * K(검색할 청크의 수) 의 결과가 Context의 길이가 됩니다.

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                               chunk_overlap=200)
# 0~1000, 800~1800, 1600~2600, ...
# 구분자(공백, 줄바꿈, 탭 등: 지정 가능)를 기준으로
# 사이즈 기준에 최대한 맞게 분리
chunks = text_splitter.split_documents(preprocessed_docs)
print(len(chunks))

1017


## Vector DB 구성하기

구성된 청크를 ChromaDB 벡터 데이터베이스에 로드합니다.   

In [30]:
from langchain_chroma import Chroma

Chroma().delete_collection() # (메모리에 저장하는 경우) 기존 데이터 삭제

# DB 구성하기
db = Chroma(embedding_function=openai_embeddings,
            persist_directory="./chroma_OpenAI",
            # 파일 시스템에 저장 (생략시 메모리에 저장)

            collection_name='Web', # 식별 이름

            collection_metadata={'hnsw:space':'l2'},
            # l2 메트릭 설정(기본값, cosine, mmr 로 변경 가능)
            )

DB에 document를 추가합니다.    
OpenAI 임베딩은 30만 토큰 동시 처리 제한이 있어, 나눠서 전달합니다.

In [31]:
from tqdm import tqdm
import time
print(len(chunks))
# 300,000 토큰 제한

# 20개씩 추가
for i in tqdm(range(0, len(chunks), 20)):
    db.add_documents(chunks[i:min(i+20, len(chunks))])
    time.sleep(5)

1017


100%|██████████| 51/51 [05:24<00:00,  6.37s/it]


db로부터 retriever를 구성합니다.

In [32]:
# Top 5 Search(기본값은 4)
retriever = db.as_retriever(search_kwargs={'k':5})
# 총 Context: 1000 * 5  = 5000

In [33]:
context = retriever.invoke("도메인 특화 언어 모델")
context

[Document(id='467d1f78-7458-4a19-877c-cb6bd37960fd', metadata={'source': 'https://n.news.naver.com/mnews/article/030/0003448907?sid=105'}, page_content='닫기 에치에프알, 통신장비 특화 독자 언어모델 개발 박준호 기자 박준호 기자 TALK 입력 2026.07.20. 오전 9:43 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 HFR에치에프알(HFR)이 차세대 통신 인프라 혁신을 이끌 통신장비 특화 언어모델(Story-LLM)을 독자 개발했다.이번에 개발된 에치에프알의 통신 특화 Story-LLM은 통신장비 및 AI 기반 통신망 운영 플랫폼과 연동해 네트워크 상태를 실시간으로 분석하고, 장애 징후와 예측 결과를 해석해 장애 원인과 조치 방안을 제시하는 통신 특화형 AI 모델이다.특히 외부 범용 대규모언어모델(LLM)과 비교해 통신장비 운영 환경에 요구되는 보안성, 응답성, 전문성 및 기술 통제력을 강화한 것이 특징이다.외부 API형 범용 LLM을 사용할 경우 장비 구성정보와 성능정보, 장애이력 등 민감한 네트워크 데이터를 외부 클라우드로 전송해야 할 가능성이 있다. 반면 에치에프알의 Story-LLM은 고객사 내부 서버와 폐쇄망 환경에서 구동할 수 있으며, 자체 개발한 시맨틱 코딩 기반 보안 표현 기술을 적용해 장비명과 포트, 채널 등 민감한 식별정보의 직접 노출과 데이터 외부 전송 경로를 최소화하도록 설계됐다.통신망 운영에 필요한 빠른 응답 성능과 분산형 AI 운영 구조도 갖췄다. 외부 API와 클라우드 서버를 거치지 않고 장비 인접 엣지 환경이나 고객사 내부 인프라에서 직접 분석을 수행하기 때문에 장애 징후가 발생하면 신속한 분석과 대응이 가능하다.향후에는 장비 단위의 신속한 판단과 경량 AI 추론을 수행하고, 중앙 

위 검색 결과를 전처리하여, LLM의 프롬프트로 넣기 위한 함수를 구성합니다.

In [34]:
from typing import Iterable, Sequence
from xml.sax.saxutils import escape
from langchain_core.documents import Document


def format_docs(
    docs: Iterable[Document],
    metadata_keys: Sequence[str] = ("source",),
) -> str:
    """
    List[Document] -> XML 직렬화 문자열.

    - 청크 경계: <document index="N"> 태그로 명시
    - 메타데이터: metadata_keys 에 지정한 키만 <meta>로 포함 (누락 키는 자동 생략)
    - 본문: XML 특수문자 escape (본문에 '<', '>' 가 있어도 경계 유지)
    """
    parts: list[str] = ["<documents>"]
    for i, doc in enumerate(docs, start=1):
        parts.append(f'  <document index="{i}">')
        for key in metadata_keys:
            value = doc.metadata.get(key)
            if value is None:
                continue
            parts.append(
                f'    <meta name="{escape(str(key))}">{escape(str(value))}</meta>'
            )
        parts.append(f"    <content>{escape(doc.page_content)}</content>")
        parts.append("  </document>")
    parts.append("</documents>")
    return "\n".join(parts)


print(format_docs(context, metadata_keys=("source", "page", "section")))

<documents>
  <document index="1">
    <meta name="source">https://n.news.naver.com/mnews/article/030/0003448907?sid=105</meta>
    <content>닫기 에치에프알, 통신장비 특화 독자 언어모델 개발 박준호 기자 박준호 기자 TALK 입력 2026.07.20. 오전 9:43 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 HFR에치에프알(HFR)이 차세대 통신 인프라 혁신을 이끌 통신장비 특화 언어모델(Story-LLM)을 독자 개발했다.이번에 개발된 에치에프알의 통신 특화 Story-LLM은 통신장비 및 AI 기반 통신망 운영 플랫폼과 연동해 네트워크 상태를 실시간으로 분석하고, 장애 징후와 예측 결과를 해석해 장애 원인과 조치 방안을 제시하는 통신 특화형 AI 모델이다.특히 외부 범용 대규모언어모델(LLM)과 비교해 통신장비 운영 환경에 요구되는 보안성, 응답성, 전문성 및 기술 통제력을 강화한 것이 특징이다.외부 API형 범용 LLM을 사용할 경우 장비 구성정보와 성능정보, 장애이력 등 민감한 네트워크 데이터를 외부 클라우드로 전송해야 할 가능성이 있다. 반면 에치에프알의 Story-LLM은 고객사 내부 서버와 폐쇄망 환경에서 구동할 수 있으며, 자체 개발한 시맨틱 코딩 기반 보안 표현 기술을 적용해 장비명과 포트, 채널 등 민감한 식별정보의 직접 노출과 데이터 외부 전송 경로를 최소화하도록 설계됐다.통신망 운영에 필요한 빠른 응답 성능과 분산형 AI 운영 구조도 갖췄다. 외부 API와 클라우드 서버를 거치지 않고 장비 인접 엣지 환경이나 고객사 내부 인프라에서 직접 분석을 수행하기 때문에 장애 징후가 발생하면 신속한 분석과 대응이 가능하다.향후에는 장비 단위의 신속한 판단과 경량 AI 추론을 수행하고, 중앙 관제 시스템에서는 복수

구성한 format_docs 함수는 이후 체인에 포함합니다.

## Prompting

RAG를 위한 간단한 프롬프트를 작성합니다.

In [35]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate([
    ("user", '''당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면,
"정보가 부족하여 답변할 수 없습니다."를 출력하세요.
---
Context: {context}
---
Question: {question}''')])

prompt.pretty_print()

================================ Human Message =================================

당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면,
"정보가 부족하여 답변할 수 없습니다."를 출력하세요.
---
Context: {context}
---
Question: {question}


## Chain

RAG를 수행하기 위한 Chain을 만듭니다.

RAG Chain은 프롬프트에 context와 question을 전달해야 합니다.    
Question을 입력받아, Context를 함께 프롬프트에 전달합니다.

In [36]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    # retriever : question을 받아서 context 검색: document 반환
    # format_docs : document 형태를 받아서 텍스트로 변환
    # RunnablePassthrough(): 체인의 입력을 그대로 저장
    | prompt
    | llm
    | StrOutputParser()
)

In [37]:
rag_chain.invoke("도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?")

'도메인 특화 언어 모델은 특정 산업·업무·지역 언어에 맞는 전문 용어와 문맥, 데이터를 집중적으로 학습해 해당 분야에서 범용 모델보다 높은 정확도와 활용성을 제공하도록 만든 언어 모델입니다. 소형 모델이라도 특정 도메인에 특화하면 더 큰 범용 모델의 성능을 능가할 수 있습니다.\n\n예시는 다음과 같습니다.\n\n- **에치에프알 ‘Story-LLM’**: 통신장비와 네트워크 운영에 특화된 모델로, 네트워크 상태를 실시간 분석하고 장애 원인과 조치 방안을 제시합니다.\n- **이스트소프트 ‘앨런 LLM 제주어 v1 4B’**: 제주어와 표준어 간 소통에 특화된 40억 파라미터 규모의 지역 언어 모델입니다.\n- **LG유플러스 ‘익시젠(ixi-GEN)’**: 통신·금융 분야의 전문 용어와 문맥을 추가 학습한 산업 특화 소형 언어 모델입니다. 도메인 특화 학습 결과 통신 분야에서 약 56%, 금융 분야에서 약 39%의 성능 향상이 나타났습니다.\n- 그 밖에 **광산 탐사 시뮬레이션, 제약사의 신약 개발, 금융사의 재무 운영** 등에 특화된 모델도 도메인 특화 모델의 사례가 될 수 있습니다.'

In [38]:
rag_chain.invoke("인공지능의 최근 발전 방식은? 관련 링크도 보여주세요")

'최근 인공지능은 다음과 같은 방향으로 발전하고 있습니다.\n\n1. **피지컬 AI와 로봇으로 확장**\n   - 실시간 영상 이해, 공간 추론, 상황 판단 능력을 높여 AI가 로봇을 직접 제어하는 방식입니다.\n   - 여러 로봇의 협업, 휴머노이드 제어, 기기 내 로컬 작동 기술도 강화되고 있습니다.\n   - 관련 링크: https://n.news.naver.com/mnews/article/018/0006340851?sid=105\n\n2. **범용 챗봇에서 코딩·기업용 AI로 이동**\n   - 단순 소비자용 챗봇보다 기업 업무와 소프트웨어 개발에 특화된 코딩 도구가 주요 경쟁 분야로 떠오르고 있습니다.\n   - 관련 링크: https://n.news.naver.com/mnews/article/374/0000525015?sid=104\n\n3. **성능뿐 아니라 가격 경쟁 강화**\n   - 빅테크 기업들이 코딩 성능을 개선하는 동시에 이용 가격을 낮추면서 가성비 중심의 모델 경쟁을 벌이고 있습니다.\n   - 관련 링크: https://n.news.naver.com/mnews/article/055/0001375586?sid=101\n\n4. **오픈소스 중심의 생태계 확장**\n   - AI 기업들은 대규모 모델을 공개해 글로벌 개발자 생태계를 확보하고 기술 표준을 선점하려 하고 있습니다.\n\n5. **모델 자체에서 에이전트와 응용 서비스로 발전**\n   - 단순히 콘텐츠를 생성하는 수준을 넘어 편집·제어·업무 수행 등 실제로 유용한 작업을 자율적으로 처리하는 AI 에이전트가 중요해지고 있습니다.\n\n6. **현실 산업과 기초과학에 적용**\n   - AI 활용 범위가 자율주행, 로봇공학, 제조업과 같은 물리적 산업뿐 아니라 수학 난제 해결 등 기초과학 분야로 확대되고 있습니다.\n   - 4~6 관련 링크: https://n.news.naver.com/mnews/article/047/0002524326?sid=101'

In [39]:
rag_chain.invoke("알리바바의 언어 모델 이름은?")

'알리바바의 언어 모델은 **Qwen 3.5(큐원 3.5)**입니다.'

assign()을 이용하면, 체인의 결과를 받아 새로운 체인에 전달하고, 그 결과를 가져옵니다.

In [40]:
# assign : 결과를 받아서 새로운 인수 추가하고 원래 결과와 함께 전달

from langchain_core.runnables import RunnableParallel

rag_chain_from_docs = (
    prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_source = RunnableParallel(
    context = retriever | format_docs, question =
    RunnablePassthrough()).assign(answer=rag_chain_from_docs)

rag_chain_with_source.invoke("인공지능의 최근 발전 방식은? 관련 링크도 보여주세요")

# retriever가 1번 실행됨
# retriever의 실행 결과를 rag_chain_from_docs 에 넘겨주기 때문에


{'context': '<documents>\n  <document index="1">\n    <meta name="source">https://n.news.naver.com/mnews/article/018/0006340851?sid=105</meta>\n    <content>닫기 구글, \'피지컬 AI\' 속도 낸다…현장 투입 앞당길 \'새 로봇 뇌\' 공개 한광범 기자 한광범 기자 입력 2026.07.31. 오전 7:31 수정 2026.07.31. 오전 7:51 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 체화 추론모델 ‘제미나이 로보틱스 ER 2’…\'지속\' 실시간 작업비디오 이해·공간지능 고도…이종로봇 간 협업·안전제어 지원 (구글 제공)[이데일리 한광범 기자] 구글이 실시간 상황 판단과 다중 로봇 협업 능력을 극대화한 신규 체화된 추론(Embodied Reasoning) 모델을 선보이며 피지컬 AI 생태계 확장에 나섰다. 연속적인 비디오 피드를 통해 작업 진행 상황을 정밀하게 추적하고 문제 발생 시 스스로 대응할 수 있도록 시간적 지능과 범용 공간 지능을 대폭 끌어올린 것이 특징이다.구글은 31일 자사 블로그를 통해 비디오 이해와 작업 오케스트레이션, 다중 로봇 협업 역량을 한층 강화한 ‘제미나이 로보틱스 ER 2(Gemini Robotics ER 2)’를 공개했다.특히 구글은 이번 발표에서 상위 추론을 담당하는 ER 2뿐만 아니라, 휴머노이드 전신과 양팔 제어에 최적화된 시각-언어-행동(VLA) 모델 ‘제미나이 로보틱스 2’, 단 몇 시간의 데이터만으로 새 로봇 폼팩터에 적응해 기기 자체에서 로컬로 작동하는 ‘제미나이 로보틱스 온디바이스 2’ 등 고성능 모델 3종을 함께 제시하며 피지컬 AI 라인업을 완성했다.스티븐 한센(Steven Hansen) 수석 소프트웨어 엔지니어와 펭 수(Peng Xu) 수석 소프트웨어 엔지니

이번에는 오픈 모델을 사용합니다.   
오픈 임베딩을 통해 구성한 DB와 원래 DB를 비교해 보겠습니다.

In [41]:
open_db = Chroma(embedding_function=open_embeddings,
                           persist_directory="./chroma_open", # 별도 폴더에 저장
                           collection_name='Web', # 식별 이름
                           collection_metadata={'hnsw:space':'l2'}
                           )

# 20개씩 추가
for i in tqdm(range(0, len(chunks), 20)):
    open_db.add_documents(chunks[i:min(i+20, len(chunks))])


100%|██████████| 51/51 [34:27<00:00, 40.53s/it]


이후는 동일합니다.

In [42]:
open_retriever = open_db.as_retriever(search_kwargs={'k':5})

In [43]:
open_retriever.invoke("도메인 특화 언어 모델")

[Document(id='2e275844-f3d9-4fa8-9e54-02d3f33b4ae0', metadata={'source': 'https://n.news.naver.com/mnews/article/014/0005475295?sid=105'}, page_content='닫기 이스트소프트, ‘제주어 특화 LLM’ 공개…국내 맞춤 AX 기반 구축 조윤주 기자 조윤주 기자 입력 2026.02.09. 오전 9:01 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 [파이낸셜뉴스] 이스트소프트가 지역 특화 언어 모델인 ‘앨런 LLM 제주어 v1 4B(Alan LLM Jeju Dialect v1 4B)’를 세계 최대의 오픈소스 플랫폼 허깅페이스에 공개했다고 9일 밝혔다. 이번 제주어 특화 LLM 공개는 국내 환경에 최적화된 기술 고도화를 통해 대국민 AX 기반을 다지기 위해 진행됐다. 특히 한국적 특성이 반영된 데이터를 정교하게 학습시켜 실제 현장에서 즉시 활용 가능한 ‘국내 맞춤형 AX’ 기술력을 확보했다는 점에서 의미가 크다. 이 모델은 오픈소스 기반의 고도화된 미세 조정(Fine-tuning) 기법을 적용해, 40억 개의 파라미터라는 효율적인 규모 내에서 제주어와 표준어 사이의 정교한 소통을 구현했다. 무엇보다 4B 규모의 경량 모델이라 하더라도 타겟 언어와 특정 도메인에 특화될 경우, 수백억 파라미터의 거대 모델 성능을 능가할 수 있음을 이번 공개를 통해 입증했다. 이스트소프트는 이러한 강력한 파인튜닝 역량을 바탕으로 향후 독자 AI 파운데이션 모델과 시너지를 창출하고, 다양한 지역 인프라와 결합해 관광·행정 등 다방면에서 실질적인 지역 특화 AX의 성공 사례를 만들어갈 방침이다. 특히 제주국제자유도시개발센터(JDC)와 ‘제주 지역의 AI 전환 및 글로벌 경쟁력 강화를 위한 업무협약’을 체결하는 등 자사의 제주 캠퍼스를 중심으로 한 활동도 

In [44]:
rag_chain_open = (
    {"context": open_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

예시 질문을 통해 정답과 검색 결과를 비교해 보겠습니다.

In [45]:
questions = ["도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?",
             "인공지능의 최근 발전 방식은? 관련 링크도 보여주세요",
             "알리바바의 언어 모델 이름은?"]

In [46]:
# Retriever 비교

contexts_hf = open_retriever.batch(questions)
contexts_openai = retriever.batch(questions)

In [47]:
for i in range(len(questions)):
    print(f"-- Question:{questions[i]}")
    oai_chunks = '\n'.join([x.page_content[:50] for x in contexts_openai[i]])
    hf_chunks = '\n'.join([x.page_content[:50] for x in contexts_hf[i]])
    print(f"OpenAI: \n{oai_chunks}")
    print(f"Hf: \n{hf_chunks}")
    print('----------')



-- Question:도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?
OpenAI: 
닫기 에치에프알, 통신장비 특화 독자 언어모델 개발 박준호 기자 박준호 기자 TALK 입력
닫기 이스트소프트, ‘제주어 특화 LLM’ 공개…국내 맞춤 AX 기반 구축 조윤주 기자 조
닫기 이스트소프트, 제주 특화 AI 언어모델 오픈소스 공개 박종진 기자 박종진 기자 입력 
국가 간 힘겨루기 수단으로 전면에 나섰다"며 "단일 모델 의존 자체가 핵심 리스크"라고 진
닫기 LGU+, 세계 최고 권위 AI 학회에 익시젠 논문 채택…“산업 특화 기술력 입증” 
Hf: 
닫기 딥서치·모티프테크놀로지스, 금융 특화 AI 모델 공동 개발 추진 입력 2026.04.
닫기 이스트소프트, ‘제주어 특화 LLM’ 공개…국내 맞춤 AX 기반 구축 조윤주 기자 조
늘고 있다"고 덧붙였다.다만 도메인 특화 모델 개발은 범용 모델 활용보다 훨씬 까다롭다. 
국가 간 힘겨루기 수단으로 전면에 나섰다"며 "단일 모델 의존 자체가 핵심 리스크"라고 진
닫기 AI 모델 최적화도 국가전략기술 인정 김윤수 기자 김윤수 기자 입력 2025.12.1
----------
-- Question:인공지능의 최근 발전 방식은? 관련 링크도 보여주세요
OpenAI: 
닫기 구글, '피지컬 AI' 속도 낸다…현장 투입 앞당길 '새 로봇 뇌' 공개 한광범 기자
인식한 언어 데이터를 함께 이해할 수 있도록 지원하는 기계학습 모델이다. 로봇이 이에 기반
5와 같은 라인에 있지만 가성비 측면에서 훨씬 강점이 있죠. 이용자 입장에서는 이 사분면에
닫기 순위 바뀐 AI 왕좌…WSJ "오픈AI 챗GPT 공들이다 앤트로픽에 추월" 김종윤 기
닫기 딥시크, 미니맥스, 바이트댄스... 동시에 터진 'AI 빅뱅' 입력 2026.08.0
Hf: 
서울 JW 메리어트 호텔에서 열린 '딜로이트 커넥트 코리아 2026' 행사장에서 진행된 그
닫기 딥시크, 미니맥스, 바이트댄스... 동시에 터진 'AI 빅뱅' 입력 2026.08.0
로봇 구동을

In [48]:
# 최종 결과 비교
oai_results = rag_chain.batch(questions)
hf_results = rag_chain_open.batch(questions)

In [49]:
for i in range(len(questions)):
    print(f"-- Question:{questions[i]}")
    print(f"OpenAI: \n{oai_results[i]}")
    print('--')
    print(f"Hf: \n{hf_results[i]}")
    print('----------')


-- Question:도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?
OpenAI: 
도메인 특화 언어 모델은 **특정 산업·업무·지역 언어에 사용되는 전문 용어, 문맥, 데이터 등을 집중적으로 학습해 해당 분야에서 높은 정확도와 활용성을 제공하는 언어 모델**입니다. 범용 모델보다 규모가 작더라도 특정 분야에서는 더 뛰어난 성능을 낼 수 있지만, 개발하려면 도메인 온톨로지 설계와 정밀한 데이터 수집·정제·라벨링 등이 필요합니다.

예시는 다음과 같습니다.

- **에치에프알 ‘Story-LLM’**: 통신장비와 네트워크 상태를 분석하고 장애 원인 및 조치 방안을 제시하는 통신장비 특화 모델입니다.
- **이스트소프트 ‘앨런 LLM 제주어 v1 4B’**: 제주어와 표준어 간 소통에 특화된 40억 개 파라미터 규모의 지역 언어 모델입니다.
- **LG유플러스 ‘익시젠(ixi-GEN)’**: 통신·금융 분야의 전문 용어와 문맥을 추가 학습한 산업 특화 소형 언어 모델로, 실험에서 통신 분야 약 56%, 금융 분야 약 39%의 성능 향상을 보였습니다.
- 그 밖에 **광산 탐사 시뮬레이션, 제약사의 신약 개발, 금융사의 재무 운영** 등에 특화된 모델도 도메인 특화 모델의 사례입니다.
--
Hf: 
도메인 특화 언어 모델(Domain-Specific LLM)은 금융·의료·법률·과학·행정처럼 특정 산업이나 업무 영역의 데이터와 전문 지식을 집중적으로 학습해 해당 분야의 과제를 수행하도록 만든 언어 모델입니다. 범용 모델보다 전문 업무에 적합하지만, 개발하려면 도메인별 온톨로지 설계, 정밀한 데이터 라벨링·수집·정제가 필요합니다.

예시는 다음과 같습니다.

- **금융 특화 LLM**: 딥서치의 금융 데이터·택소노미와 모티프테크놀로지스의 파운데이션 모델을 결합한 모델로, 기업 분석·가치평가 보고서 생성, 자산운용 지원, 포트폴리오 위험 감시, M&A 대상 분석 등에 활용됩니다.
- **제주어 특화 LLM**: 이스트소프트의 **‘앨런 LLM 제주어 v1